# Mold Data — Quality Issue Summary by Family

Processes `data/mold_data.json` and exports an Excel report.

**Issues covered:** I1 · I5 · I7 · I8 · I9 · I11 · I12 · I13 · I14

**Output:** `data/dq_issues_by_family.xlsx`  
- Sheet 1 `Flag Matrix` — instance counts per family × issue, sorted by most issues first  
- One sheet per issue — affected families with per-component detail

## 1. Imports

In [ ]:
import json

import pandas as pd

## 2. Load Data

In [ ]:
with open('../data/mold_data.json', encoding='utf-8') as f:
    data = json.load(f)

families   = data['families']
vendor_ref = data['reference']['vendors']

print(f'Loaded {len(families)} mold families, {len(vendor_ref)} vendors')

## 3. Issue Detection Functions

In [ ]:
def check_null_vendor_id(family):
    """I1 — mold has null vendorId"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            for i, mold in enumerate(comp.get('molds', [])):
                if mold.get('vendorId') is None:
                    hits.append(f'{st} > {ck} > mold[{i}]')
    return hits


def check_null_daily_output(family):
    """I5 — mold has null dailyOutput"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            for i, mold in enumerate(comp.get('molds', [])):
                if mold.get('capacity', {}).get('dailyOutput') is None:
                    hits.append(f'{st} > {ck} > mold[{i}]')
    return hits


def check_all_sizing_null(family):
    """I7 — every moldSizeToShoeSizes value is null (no shoe-size mapping defined)"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            sizing = comp.get('sizingRules', {}).get('moldSizeToShoeSizes', {})
            if sizing and all(v is None for v in sizing.values()):
                hits.append(f'{st} > {ck}')
    return hits


def check_sizing_missing_from_qty(family):
    """I8 — sizingRules lists sizes absent from qtyByMoldSize"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            sizing_keys = set(comp.get('sizingRules', {}).get('moldSizeToShoeSizes', {}).keys())
            for i, mold in enumerate(comp.get('molds', [])):            
                qty_keys = set(mold.get('inventory', {}).get('qtyByMoldSize', {}).keys())
                missing = sizing_keys - qty_keys
                if sizing_keys and qty_keys and missing:
                    hits.append(f'{st} > {ck} > mold[{i}]: missing {sorted(missing)}')
    return hits


def check_total_qty_mismatch(family):
    """I9 — totalMoldQty != sum(qtyByMoldSize values)"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            for i, mold in enumerate(comp.get('molds', [])):
                inv = mold.get('inventory', {})
                total = inv.get('totalMoldQty')
                qty_by_size = inv.get('qtyByMoldSize', {})
                if total is not None and qty_by_size:
                    computed = sum(v for v in qty_by_size.values() if v is not None)
                    if computed != total:
                        hits.append(
                            f'{st} > {ck} > mold[{i}]: declared={total}, actual_sum={computed}'
                        )
    return hits


def check_style_soletype_missing(family):
    """I11 — style lists a soleType that has no component block in this family"""
    hits = []
    comp_sole_types = set(family.get('components', {}).keys())
    for brand, styles in family.get('stylesUsingThisFamily', {}).items():
        for style in styles:
            for st in style.get('soleTypes', []):
                if st not in comp_sole_types:
                    hits.append(
                        f"{brand} > {style.get('styleName')}: '{st}' missing from components"
                    )
    return hits


def check_vendor_location_mismatch(family, vendor_ref):
    """I12 — vendor shortName country suffix contradicts location.code"""
    COUNTRY_CODES = {'BD', 'CN', 'VN', 'ID', 'KH', 'TH', 'MM', 'IN', 'PK', 'PH', 'TW', 'KR'}
    hits = []
    seen_vendors = set()
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            for i, mold in enumerate(comp.get('molds', [])):
                vid = mold.get('vendorId')
                if not vid or vid not in vendor_ref or vid in seen_vendors:
                    continue
                seen_vendors.add(vid)
                vdata = vendor_ref[vid]
                loc_code = (vdata.get('location') or {}).get('code', '')
                short_name = vdata.get('shortName', '')
                suffix = short_name.split('-')[-1][:2].upper() if '-' in short_name else ''
                if suffix in COUNTRY_CODES and suffix != loc_code:
                    hits.append(
                        f"{st} > {ck} > mold[{i}]: vendor {vid} "
                        f"shortName='{short_name}' but location='{loc_code}'"
                    )
    return hits


def check_duplicate_molds(family):
    """I13 — duplicate (vendorId, moldShopCode, initSeason) within the same component"""
    hits = []
    for st, comps in family.get('components', {}).items():
        for ck, comp in comps.items():
            seen = {}
            for i, mold in enumerate(comp.get('molds', [])):
                key = (mold.get('vendorId'), mold.get('moldShopCode'), mold.get('initSeason'))
                if key in seen:
                    hits.append(
                        f'{st} > {ck}: mold[{seen[key]}] duplicates mold[{i}] — {key}'
                    )
                else:
                    seen[key] = i
    return hits


def check_orphan_family(family):
    """I14 — no style references this mold family"""
    if not family.get('stylesUsingThisFamily'):
        return ['No styles reference this family']
    return []

## 4. Build Per-Family Summary

In [ ]:
CHECKS = [
    ('I1_null_vendorId',          lambda f: check_null_vendor_id(f)),
    ('I5_null_dailyOutput',        lambda f: check_null_daily_output(f)),
    ('I7_all_sizing_null',         lambda f: check_all_sizing_null(f)),
    ('I8_sizing_missing_from_qty', lambda f: check_sizing_missing_from_qty(f)),
    ('I9_total_qty_mismatch',      lambda f: check_total_qty_mismatch(f)),
    ('I11_style_stype_missing',    lambda f: check_style_soletype_missing(f)),
    ('I12_vendor_loc_mismatch',    lambda f: check_vendor_location_mismatch(f, vendor_ref)),
    ('I13_duplicate_molds',        lambda f: check_duplicate_molds(f)),
    ('I14_orphan_family',          lambda f: check_orphan_family(f)),
]

ISSUE_LABELS = {
    'I1_null_vendorId':          'I1 · Null vendorId',
    'I5_null_dailyOutput':        'I5 · Null dailyOutput',
    'I7_all_sizing_null':         'I7 · All sizing null',
    'I8_sizing_missing_from_qty': 'I8 · Sizing↔Qty mismatch',
    'I9_total_qty_mismatch':      'I9 · totalMoldQty mismatch',
    'I11_style_stype_missing':    'I11 · Style soleType missing',
    'I12_vendor_loc_mismatch':    'I12 · Vendor location mismatch',
    'I13_duplicate_molds':        'I13 · Duplicate molds',
    'I14_orphan_family':          'I14 · Orphan family',
}

SHEET_NAMES = {
    'I1_null_vendorId':          'I1 - Null vendorId',
    'I5_null_dailyOutput':        'I5 - Null dailyOutput',
    'I7_all_sizing_null':         'I7 - All sizing null',
    'I8_sizing_missing_from_qty': 'I8 - Sizing-Qty mismatch',
    'I9_total_qty_mismatch':      'I9 - TotalQty mismatch',
    'I11_style_stype_missing':    'I11 - Style soleType miss',
    'I12_vendor_loc_mismatch':    'I12 - Vendor loc mismatch',
    'I13_duplicate_molds':        'I13 - Duplicate molds',
    'I14_orphan_family':          'I14 - Orphan family',
}

rows = []
for family_code, family in families.items():
    row = {'family': family_code, 'total_issues': 0}
    for col, fn in CHECKS:
        hits = fn(family)
        row[col] = len(hits)
        row[col + '_detail'] = ' | '.join(hits) if hits else ''
        row['total_issues'] += (1 if hits else 0)
    rows.append(row)

df = pd.DataFrame(rows).set_index('family')
count_cols = [c for c, _ in CHECKS]

print(f'Families with at least one issue: {(df["total_issues"] > 0).sum()} / {len(df)}')

## 5. Export to Excel

In [ ]:
output_path = '../data/dq_issues_by_family.xlsx'

# Flag matrix: families with issues, sorted by most issue types hit
_sorted_idx = (
    df[df['total_issues'] > 0]
    .sort_values('total_issues', ascending=False)
    .index
)
flag_df = (
    df.loc[_sorted_idx, ['total_issues'] + count_cols]
    .rename(columns={'total_issues': 'Total Issues', **ISSUE_LABELS})
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1 — flag matrix
    flag_df.to_excel(writer, sheet_name='Flag Matrix')

    # One sheet per issue (skipped if no families are affected)
    for col, _ in CHECKS:
        affected = df[df[col] > 0][[col, col + '_detail']].copy()
        if affected.empty:
            continue
        affected = (
            affected
            .rename(columns={col: 'instance_count', col + '_detail': 'detail'})
            .sort_values('instance_count', ascending=False)
        )
        affected.index.name = 'family'
        affected.to_excel(writer, sheet_name=SHEET_NAMES[col])

print(f'Exported: {output_path}')
print(f'  Flag Matrix : {len(flag_df)} families')
for col, _ in CHECKS:
    n = (df[col] > 0).sum()
    if n:
        print(f'  {SHEET_NAMES[col]:<28}: {n} families')